# Extract model values at ICES locations

* [X] Load regions
* [X] Load model
* [X] Deal with interpolation
* [X] parquet output


## Load regions

In [1]:
import xarray as xr
import matplotlib.pyplot as plt 

from  matplotlib.colors import LogNorm
from matplotlib.ticker import LogFormatter 

import numpy as np
import pandas as pd

from cmocean import cm
from matplotlib.colors import ListedColormap

from cartopy import crs as ccrs
import cartopy.feature as cf


import datetime

In [2]:
# User Inputs #
modyear = 2010

datadir = '/home/cvao/proj_codeblue/data/validation/'

regionfile = '/home/cvao/postprocessing-toolbox/NetCDF_manipulation/Geospatial/Bathymetry_NoS_with_regions_AC_RED.nc'
regionfile = '/home/cvao/postprocessing-toolbox/NetCDF_manipulation/Geospatial/Bathymetry_NoS_with_regions_AC.nc'

modeldir = '/ec/res4/hpcperm/cvao/BGC/OUTPUTS/nos2_1/%s/'%modyear
modelfile= modeldir + 'NoS_1.tsout3.nc'
figsuffix = 'nos2_1'



In [3]:
xreg = xr.load_dataset(regionfile)
nreg = (xreg.region_id.max().values+1).astype(int)

if False:
    print(nreg)
    
    regcmap = ListedColormap(plt.cm.Spectral(np.linspace(0, 1, nreg)))
    
    fig = plt.figure(figsize = (15,8))
    ax= fig.add_subplot(111, projection =ccrs.PlateCarree())
    xreg.region_id.plot(cmap=regcmap)
    
    ax.gridlines()
    ax.coastlines(color='k')
    # Countries' borders
    ax.add_feature(cf.BORDERS, color = 'darkgrey')
    ax.add_feature(cf.RIVERS, color = 'darkblue')



In [4]:
## Loading Model data    
xmod = xr.open_dataset(modelfile)
xmod['chl']=(xmod['chlsc']+xmod['chlnsc'])


In [9]:
for c in xmod.coords:
    print(c)



lon
lat
lev
time


## Reading ICES Data

In [10]:

yrs='2010_2015'

vars=['oxy','nox','nh4','chl','po4','sio']

for var in vars:

    # Shaping local in situ dataframe 
    dfl =pd.read_parquet(datadir+'%s_%s.parquet'%(var,yrs))
    dflt=dfl[dfl['datetime'].dt.year == modyear]
    dflt = dflt.dropna()

    # Get coordinaates
    lons  = xr.DataArray(dflt['lon'], dims="points")
    lats  = xr.DataArray(dflt['lat'], dims="points")
    times = xr.DataArray(dflt['datetime'], dims="points")
    depths = xr.DataArray(dflt['depth'], dims="points")

    # Get Region ID
    regs = xreg['region_id'].interp({'lon':lons, 'lat':lats})
    dflt['reg']=regs

    # Some plots if details needed
    if False:
        regdatacount = dfl.groupby('reg').count()['lon'].values
        plt.bar(dfl['reg'].unique(), regdatacount)
        plt.xlabel('Regions')
        plt.ylabel('Data counts')

    if False:
        ## -> To check regionalisation
        plt.scatter(regs['lon'],regs['lat'],50, regs, cmap=regcmap)

    #####
    # MODEL
    
    if var == 'chl':
        mvar='chl'
    if var == 'nox':
        mvar='no3'
        conversion = 1000
    if var == 'nh4':
        mvar='nh4'
        conversion = 1000
    elif var=='oxy':
        mvar='o2'
        conversion = 1000/44.66 # molO2/m³ -> ml/l

    # Local model data
    xmodv = xmod[[mvar, 'depth']].copy()*conversion
    
    xmodv = xmodv.chunk({
        "time": 50,
        "lat": 100,
        "lon": 100,
        "lev": -1   # keep full vertical column
    })

    # First horizontal interpolation, to get vertical columns
    tmp = xmodv.interp(lon=lons, lat=lats, time=times)
    z = tmp['lev'] * tmp['depth']
    tmp = tmp.chunk({"lev": -1})
    z   = z.chunk({"lev": -1})
    
    # Organize the distribution of vertical interpolation 
    result = xr.apply_ufunc(
        np.interp,
        depths,
        z,
        tmp[mvar],
        input_core_dims=[[], ["lev"], ["lev"]],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[tmp[mvar].dtype],
    )

    # This is where the computation actually takes place. All the above is 'lazy'
    dflt['mod']=result

    dflt.to_parquet("%s/ICES_%s_%s.parquet"%(modeldir,modyear,var))

In [11]:
dflt

,datetime,lon,lat,depth,sio,sio_qv,reg,mod
0,2010-01-25 21:04:00+00:00,3.008,51.580,3.0,13.30,0,3.0,7438.627401
1,2010-01-26 01:05:00+00:00,2.700,51.750,3.0,5.20,0,3.0,1212.041714
2,2010-01-26 02:23:00+00:00,2.420,51.680,3.0,5.70,0,3.0,1124.866280
3,2010-01-26 22:51:00+00:00,2.808,51.420,3.0,14.30,0,3.0,6016.761159
4,2010-01-27 00:59:00+00:00,2.468,51.263,3.0,3.30,0,3.0,5459.215908
...,...,...,...,...,...,...,...,...
382370,2010-05-02 09:24:00+00:00,-2.234,59.283,10.0,1.57,0,NaN,NaN
382371,2010-05-02 09:24:00+00:00,-2.234,59.283,18.0,1.54,0,NaN,NaN
382372,2010-05-02 09:24:00+00:00,-2.234,59.283,29.0,1.62,0,NaN,NaN
382373,2010-05-02 09:24:00+00:00,-2.234,59.283,48.0,1.55,0,NaN,NaN
